# README

- **Author**: RUEI-HSIANG, CHANG／張睿庠（B12303030；AI協助）
- **Created At**: 2026-11-08
- **Last Modified At**: 2026-11-08

---
## What does this file do?
整合11/4主表與楊喬茵兩份補查表；保留舊主表。採張睿庠確認的G5修訂規則與公開呈現學歷分類。兩位同名歷史連結再核對，詳見 report/clustering_readiness_20261108.md。姓名、地區加上職涯／學校等交叉支持；不宣稱戶籍認證。未查者維持未知，G6僅在已執行指定搜尋且未發現符合親屬時編0；未搜尋或有待查線索仍空白。照片推斷性別不採用。

---
## What does this file take?
- **Source Data Sets**：下方PINS逐檔固定SHA-256。主表文件為data/processed/g1_g6_20261103_documentation.ipynb；兩補查表文件為data/code/mayor_gaps_20261104.ipynb、data/code/council_tk_gaps_20261104.ipynb。新來源與閱讀限制見report。

---
## What does this file output?
三份data/processed/clustering_{all,mayor,council}_20261108.csv（collection / clustering candidate inputs，尚非全欄完整分析樣本），另有逐格merge_audit、readiness、remaining_worklist與G6新增證據四表。各精確路徑、schema、筆數與SHA由最後cell列出；文件data/processed/clustering_20261108_documentation.ipynb。三池是全體、首長、議員，不按縣市各自分群。保留登記名單，未將最終資格審核視為完成。沒有補值、刪樣本或執行分群。


2026-11-08魏造文續查：選舉黃頁明載男、API gender=m，補G2；完成指定親屬線上搜尋無符合證據，依使用者規則補G6=0。生日／年齡仍未知。詳見report/wei_zaowen_investigation_20261108.md。

In [1]:
from pathlib import Path
import pandas as pd, hashlib, json, csv, re
from IPython.display import display
ROOT = next(x for x in [Path.cwd(), *Path.cwd().parents] if (x/'AGENTS.md').exists() and (x/'templates').is_dir())
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
PINS = {'data/processed/candidates_g1_g6_current.csv': 'd8695b306bd7eadbf0de5c8bb2f368c717d23d4ef7ae55cf3cf4b8fa1afbb59c', 'data/processed/mayor_g1_g6_gap_review_20261104.csv': 'b5de33d2ae913e41e8f3f4ecb08855b4066e146deb05c64ed3ab77809ee562ca', 'data/processed/council_tk_g1_g5_gap_review_20261104.csv': 'ff0233b49349a37616895981fb797f929ed8512c2afdab874b041237aa39a82a', 'data/raw/cec_all_offices/112_elcand.csv': 'f355705254d5ca457d4fe130b77983b769602126b10713e7c968889d2d52530c', 'data/raw/cec_all_offices/105_elcand.csv': '0b4ab7af1068906c1579eaedcc81bf022c096953a768ba2a848412bded4f8408', 'data/raw/identity_recheck_20261108/chen_moi.html': '528c252ef587198531def9debed27dc609ea18d96de43096ff6a9d401628a2ca', 'data/raw/identity_recheck_20261108/chen_udn.html': 'c26cbc54c331a72bfd12691a9e159545fdd679c26a4a0a60b8444ffdb298557d', 'data/raw/identity_recheck_20261108/lin_bulletin.pdf': '315fc8e03910f7a6c3fcde973e2539e06949e55bf6419ecdcfca27781cc091fc', 'data/raw/identity_recheck_20261108/chen_bulletin.pdf': '34b52e9ad66d128a4f64205206d9ae05b3ab52928830330d1048814400c591d9', 'data/raw/council_tk_gaps_20261104/p08_politicaltw.html': 'bb9059d0c587719d27ee59db32dfa4757317bd01ce0cfcc3a52935016290e99e'}
for f, expected in PINS.items(): assert sha(ROOT/f)==expected, f
base=pd.read_csv(ROOT/"data/processed/candidates_g1_g6_current.csv",dtype=str,keep_default_na=False)
assert len(base)==1583 and base.candidate_key.is_unique
merged=base.copy().set_index('candidate_key',drop=False)
merged['G3_public_presentation_rule']='existing_evidence_retained'
merged['G3_unpublished_lowest_flag']='0'
merged['identity_recheck_20261108']=''
merged['supplement_20261108']=''
updates=[]
COLS={'G1':'G1_age_2026_minus_birth_year','G2':'G2_sex','G3':'G3_education_level','G4':'G4_party','G5':'G5_first_time_any_elected_office','G6':'G6_political_family'}

PREVIOUS_OUTPUT_SHA256 = {'data/processed/clustering_all_20261108.csv': '0a18735e67c59a838202f762adfda3e41db8a37cef26909b8c5c940974bf8e1f', 'data/processed/clustering_mayor_20261108.csv': '10446337c157d186cd92efab4b690f1850b872aa8c2cc0467417acb2b0e1b2ef', 'data/processed/clustering_council_20261108.csv': 'f84619e86a7e5d5233f6f3bf436c6ba8cb11a63afaed165685438fedeade9aac', 'data/processed/clustering_merge_audit_20261108.csv': 'd1a290af5659126cc974ae5d126f08deed8ad7467f45ef4348b2c15ed709235b', 'data/processed/clustering_readiness_20261108.csv': '28fd181ef3299d4cae142b8f449cbfaac128ac1d48e6620a66870007dfffaf9a', 'data/processed/clustering_remaining_worklist_20261108.csv': '149921b2eb5684de2878d3063009226cbf19849c3d48e44fddd5ad41d649abe9'}

PINS.update({'data/raw/identity_recheck_20261108/g6_hsieh.html': '27895eba249cfa42aaad4d1ad9214501d15f278114fa73f168c507729ae66fb5', 'data/raw/identity_recheck_20261108/g6_hsieh.txt': 'fda3aa5954d255c13978aea765b7f8e07004820ae276a8da72dfd4e7ce688b90', 'data/raw/identity_recheck_20261108/g6_huang.html': '7212c76b2d0029ccebbf5025f14c2dfc6480de72a38bde3a2e22adbcc9667c22', 'data/raw/identity_recheck_20261108/g6_huang.txt': '9a66d73a6377422299b8177161c1e52fe5c6878a68f1c7c0c682135a1d026ca2', 'data/raw/identity_recheck_20261108/g6_chen.html': '23501da9b9642ddb9c4d06e2aa6c7b68d36d99aba000e336df195ba0a06175e9', 'data/raw/identity_recheck_20261108/g6_chen.txt': '4355d90d918a93df76adc34c374fad253cbb68aeccbc6263ebb29e11898f81ef', 'data/raw/identity_recheck_20261108/g6_wu.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/g6_wu.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/g6_manifest.json': '389c1e9fd16101b69c4034f96ec185e3c1f2cd01beb89bf4181c9a676a97b764', 'data/raw/identity_recheck_20261108/g6_reviewed_facts.json': '8daf2b5722b547ecbb29ef513c739772aea832c10c929c966fb8536e0516ea42'})
for f,expected in PINS.items(): assert sha(ROOT/f)==expected,f

PREVIOUS_OUTPUT_SHA256.update({'data/processed/clustering_all_20261108.csv': '656944c585b6f2ebf60c4b538f84ba1c2c348e46fda4cce70272ad975ae3a967', 'data/processed/clustering_mayor_20261108.csv': '10446337c157d186cd92efab4b690f1850b872aa8c2cc0467417acb2b0e1b2ef', 'data/processed/clustering_council_20261108.csv': 'd24716afb00a20377701c4f0efcf5fbe3182c570876e857742632c66abef9bb3', 'data/processed/clustering_merge_audit_20261108.csv': '759a5def5176573b10aa2e728136062f1be086f5ce0838ec08463b52c64c2fd9', 'data/processed/clustering_readiness_20261108.csv': '8e017cc6ace259a50e117cda1cccaa42e5ce8c30856e6631e5bb571920c891dc', 'data/processed/clustering_remaining_worklist_20261108.csv': '70d32413762f0987eecb563be036a881acbb39feab0899bfebceb24edbdab3a2'})

PINS.update({'data/raw/identity_recheck_20261108/g6_search_decisions.json': 'c5c340d222b858bd84beb086c06788def74003b00d03f191623a38a140466ad8', 'data/raw/identity_recheck_20261108/g6_negative_search.txt': '352f18c47590c89b363cb04859111f6ec7ac75af325068e8503d2afe58edee52', 'data/raw/identity_recheck_20261108/g6_initial_search.txt': '92616a5b426d3363f37addbc7e05c8a15c5c249bd25c23b62196796b2b0dc6d3'})
for f,expected in PINS.items(): assert sha(ROOT/f)==expected,f

PREVIOUS_OUTPUT_SHA256.update({'data/processed/clustering_all_20261108.csv': 'fac12c144f7abf272b702ab59a66ca3ef897eebc533d13498f87fe14aba528a0', 'data/processed/clustering_mayor_20261108.csv': 'd5dbd1c66ee3b8e74effcbcafe7cbfa9b09b4f805a904c0ad4da8d9178e00223', 'data/processed/clustering_council_20261108.csv': 'd24716afb00a20377701c4f0efcf5fbe3182c570876e857742632c66abef9bb3', 'data/processed/clustering_merge_audit_20261108.csv': 'a7718ba60a49feeccae1ecfb588137ddbe0480c249923a630c857e5e629d034f', 'data/processed/clustering_readiness_20261108.csv': '2a927325a5320ba287b7e7c26b56f86f7bcebf7dcac50bae1c17097d3611fad8', 'data/processed/clustering_remaining_worklist_20261108.csv': '596dc251574a327080bf3ed1f8767a29f20fe4eddd2879dc3be6f14a7ca2a9d3', 'data/processed/clustering_g6_new_evidence_20261108.csv': '27156ef1eb4c76dd53e2a57523314fe5882eb52fa5ea1a41903b340f61812579'})

PINS.update({'data/raw/identity_recheck_20261108/lin_campaign.html': '49491740371f299ea945eca8b12c07f7ed173bdcad63867a1228bfdce183bb34', 'data/raw/identity_recheck_20261108/chen_moi.html': '528c252ef587198531def9debed27dc609ea18d96de43096ff6a9d401628a2ca', 'data/raw/identity_recheck_20261108/chen_moi.txt': '22acf7e573ffc78d6ba71302299070bf8c448bcd2437eb9f34817c51c47359dd', 'data/raw/identity_recheck_20261108/chen_udn.html': 'c26cbc54c331a72bfd12691a9e159545fdd679c26a4a0a60b8444ffdb298557d', 'data/raw/identity_recheck_20261108/chen_udn.txt': 'b055ca0c23f8c8b0094dc8a058343a132ccaa833af1e1c27fefc76a5276e5243', 'data/raw/identity_recheck_20261108/chen_maolin.html': 'd000e734e3f8143122c75a7786bb085e917c4e8c12dbbd27e62fec6e567c7b1f', 'data/raw/identity_recheck_20261108/chen_maolin.txt': 'be6feff1ce34abece9fc8710ea7c7ecaccddb13efe94b73a079715e23d0027f7', 'data/raw/identity_recheck_20261108/lin_bulletin.pdf': '315fc8e03910f7a6c3fcde973e2539e06949e55bf6419ecdcfca27781cc091fc', 'data/raw/identity_recheck_20261108/chen_bulletin.pdf': '34b52e9ad66d128a4f64205206d9ae05b3ab52928830330d1048814400c591d9', 'data/raw/identity_recheck_20261108/manifest.json': '0ff7157d45a2c22014f7892ab6e8e1e4891636d24874543eb3aaa580138f0d21', 'data/raw/identity_recheck_20261108/review_observations.json': '34f8284ac91178655eb6bc2e95633d8c6f67e4d3f2bff31024a0731c38e65458', 'data/raw/identity_recheck_20261108/g6_hsieh.html': '27895eba249cfa42aaad4d1ad9214501d15f278114fa73f168c507729ae66fb5', 'data/raw/identity_recheck_20261108/g6_hsieh.txt': 'fda3aa5954d255c13978aea765b7f8e07004820ae276a8da72dfd4e7ce688b90', 'data/raw/identity_recheck_20261108/g6_huang.html': '7212c76b2d0029ccebbf5025f14c2dfc6480de72a38bde3a2e22adbcc9667c22', 'data/raw/identity_recheck_20261108/g6_huang.txt': '9a66d73a6377422299b8177161c1e52fe5c6878a68f1c7c0c682135a1d026ca2', 'data/raw/identity_recheck_20261108/g6_chen.html': '23501da9b9642ddb9c4d06e2aa6c7b68d36d99aba000e336df195ba0a06175e9', 'data/raw/identity_recheck_20261108/g6_chen.txt': '4355d90d918a93df76adc34c374fad253cbb68aeccbc6263ebb29e11898f81ef', 'data/raw/identity_recheck_20261108/g6_wu.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/g6_wu.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/g6_manifest.json': '389c1e9fd16101b69c4034f96ec185e3c1f2cd01beb89bf4181c9a676a97b764', 'data/raw/identity_recheck_20261108/g6_reviewed_facts.json': '8daf2b5722b547ecbb29ef513c739772aea832c10c929c966fb8536e0516ea42', 'data/raw/identity_recheck_20261108/g6_negative_search.txt': '352f18c47590c89b363cb04859111f6ec7ac75af325068e8503d2afe58edee52', 'data/raw/identity_recheck_20261108/g6_initial_search.txt': '92616a5b426d3363f37addbc7e05c8a15c5c249bd25c23b62196796b2b0dc6d3', 'data/raw/identity_recheck_20261108/g6_search_decisions.json': 'c5c340d222b858bd84beb086c06788def74003b00d03f191623a38a140466ad8', 'data/raw/identity_recheck_20261108/education_current_leads.json': '42f27f5f244224b4e3a92f04675a327f1c766baca90b9a526a607fbf613a9a27', 'data/raw/identity_recheck_20261108/edu_2bc6b319e633.html': 'aae12b213bff16b957234878c496893c26c9378c451d26fe67008bbe66e6f0ed', 'data/raw/identity_recheck_20261108/edu_02e0facb3e5f.html': '0dc323fbf6fbe0c96aea29b7461e2755378c6ca4bbaf86b5864803b8de8f1b37', 'data/raw/identity_recheck_20261108/edu_2bc6b319e633.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_02e0facb3e5f.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_12cecd3ebeb1.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_12cecd3ebeb1.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_b4920d182156.html': '08dc6e1d13c1bd364ee788fddbe0654dfc0fbf564c4f302402c493718abdb59e', 'data/raw/identity_recheck_20261108/edu_b4920d182156.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_3f9e9546e799.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_3f9e9546e799.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_0f42bc6d5453.html': 'ab71e212f51569075d6e9fdcf73f3c59c6a0dffde69b62be68db65bafca461cc', 'data/raw/identity_recheck_20261108/edu_0f42bc6d5453.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_e01e7795c798.html': '9b8196bab7a2cbb8316fd5a63efcd22cd756c19ff3cf1b5b62fcfa4df9538659', 'data/raw/identity_recheck_20261108/edu_e01e7795c798.txt': '249f34e42cdd4022f45f4c32643ed4e657188dbfda4778376dc822bde6b023b4', 'data/raw/identity_recheck_20261108/edu_73e37037c5fc.html': '586007dcfcfb07f7879fc3bdf7adf9723529f928155a0355628051cc919815e2', 'data/raw/identity_recheck_20261108/edu_73e37037c5fc.txt': 'f39584005d363f1d0f65b941b64a39d04b57e9d43f4a10d762cac956242658e9', 'data/raw/identity_recheck_20261108/edu_5bb27d08fa3f.html': '91f512c4682abdd14ee71e6393626586b95fd4cefe5b678c23e8a322efb1531f', 'data/raw/identity_recheck_20261108/edu_5bb27d08fa3f.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_0fa9533d4406.html': '77c409b31288066486cfb54fc91a0b4990dd4363397247ad515d7bc7ac07adfd', 'data/raw/identity_recheck_20261108/edu_0fa9533d4406.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_9772611b4f6d.html': '95dd66e210240f219649df8ec78c2604a5debf76991b602ddde1e31428e809fa', 'data/raw/identity_recheck_20261108/edu_9772611b4f6d.txt': 'f9ceb5d69a8b8b01b36f9bb7d52ef8f7d1afe31a40888d3db22b1bf6072b0509', 'data/raw/identity_recheck_20261108/edu_ba5741600503.html': 'fc4fbc73e10b4f62c08c211fcb81166900772abf271c0cac2e146da77c547d5e', 'data/raw/identity_recheck_20261108/edu_ba5741600503.txt': '9cbeb35a514f424bcfc58ce980f53e79db6890b98b5da8720510c1990bc0b215', 'data/raw/identity_recheck_20261108/edu_9a4847c4a691.html': 'f44df2bed0172023ddbada6cc4a7a52b37af0f720903b97d42039a42334c74f6', 'data/raw/identity_recheck_20261108/edu_9a4847c4a691.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_c61bfdd11e0e.html': '6dae9b69e03ba24490c1b2503002949bbcee6b3bf6a34032ee24802625195de4', 'data/raw/identity_recheck_20261108/edu_c61bfdd11e0e.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_ed5d2b937362.html': 'a3ff2102565a9f4a1e9089df328fef52b107e88e1b04456dda2fd9fd48904a7a', 'data/raw/identity_recheck_20261108/edu_ed5d2b937362.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_aee3c94631f3.html': '5c6100eb2a358db342b8bde30a2a681bd156d011d5b8057f3227c7e4a4a0df23', 'data/raw/identity_recheck_20261108/edu_aee3c94631f3.txt': 'a1dd620d45a92c843b87b95eb4d863e60f568eda9bf745ed2f9d6aeec06f445b', 'data/raw/identity_recheck_20261108/edu_2eb4fc26b165.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_2eb4fc26b165.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_d9b8f363d586.html': 'ee3ea390f1c83277b4e5007b61c1fe4aaeaff879077969e81c92b731fccfc168', 'data/raw/identity_recheck_20261108/edu_18820697dd23.html': 'd8ffbbe403b0ee7390b74114c306c278e9b0a90529d05897ce1a299d05d57b08', 'data/raw/identity_recheck_20261108/edu_18820697dd23.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_d9b8f363d586.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_e25294c9c53c.html': '57fe4315ee5ddbfdd31b6fcf8559aac8a613d9258edc3e0b3b37258ef788ef3b', 'data/raw/identity_recheck_20261108/edu_e25294c9c53c.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_904051aa975f.html': 'a9fe5a6e29183e4eebd8fc2b705fbfdbf97e8563b276d64bbd9cd668ca32c3dd', 'data/raw/identity_recheck_20261108/edu_904051aa975f.txt': '70fc76941538593291f78e5a1243ec4d7d905bd0256f916d7490bd8000bf6b88', 'data/raw/identity_recheck_20261108/edu_5b0285ce8dc0.html': '59d924be007248e4ddc1d441a5fd45237d1670ef518b00840be33cd5de4a4883', 'data/raw/identity_recheck_20261108/edu_5b0285ce8dc0.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_dc5bbe3a0bcf.html': 'a24c6bb65b64ec2c6d6958e6c67cf8140f48da93c2a6e25e5859b0d70b761c57', 'data/raw/identity_recheck_20261108/edu_dc5bbe3a0bcf.txt': '5754f3d5e7337ba29c64376d1856c266114b0641c136449ebe134253597d5506', 'data/raw/identity_recheck_20261108/edu_c8949cc45fbd.html': '63b66367119e5ff676c59ee6ae9e9ec7be4b914baa5bdcd6a33506029f15d039', 'data/raw/identity_recheck_20261108/edu_c8949cc45fbd.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_d1d48715956b.html': 'a6960c3e0e441587a6b3646257e960c917f276cb5fc37772ad901a6cdba8cf7c', 'data/raw/identity_recheck_20261108/edu_d1d48715956b.txt': 'd88c1e30ceb5135a227d4481e4770307a3577dff64c7fccc1c5715ef2e6a9365', 'data/raw/identity_recheck_20261108/edu_5ec760d5cb86.html': '0ae1a6c3c7700d5ca7cb56eef45bbe43eeb6065496c7c1926bc60c4f8c708ad1', 'data/raw/identity_recheck_20261108/edu_743bec441d61.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_743bec441d61.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_8a931f7bc6ba.html': '7f108ba3d2cc3a5804ce85dda38b9f9fb61a6a397e5dc45e2a63066fe2b72477', 'data/raw/identity_recheck_20261108/edu_8a931f7bc6ba.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_04d34aeeacf6.html': 'e3a476b77f7a8db1758998458349a10f441aaea3b5751501bcad75ca416b241e', 'data/raw/identity_recheck_20261108/edu_04d34aeeacf6.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_3779b8d372c1.html': '8fbc2b736410669167198a35508b3b1f6de52ee742a2db12edeaf0139393c497', 'data/raw/identity_recheck_20261108/edu_3779b8d372c1.txt': '665aa0864acb9c39d067e9d6af0fee96eed21d5c5daeb71c8c55e5f7216d2358', 'data/raw/identity_recheck_20261108/edu_07ba48c85e85.html': '97d3e168b78e0f9ecefd8f9a3dbe836ea4b19d01eb09d038da6a0b9235c391e2', 'data/raw/identity_recheck_20261108/edu_3fe405b88e8c.html': '6480d8bb240d0b5749c626c7ac6cbe42779b075a22568bcff0a0cd4c42f65dc6', 'data/raw/identity_recheck_20261108/edu_07ba48c85e85.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_3fe405b88e8c.txt': 'e8bcd5f4ef9abe39f74e363aecd194756bdc528cad042096c1528cb584305692', 'data/raw/identity_recheck_20261108/edu_0cd62bab8d30.html': '8706be0d94b0e7aa125074c48dd9dae5b4bedd214aa11c04566a7190e45a72d0', 'data/raw/identity_recheck_20261108/edu_4b6358aa813c.html': '3a0a12f32ea6084424f619dd59d12cb56d857b2b24ac3fa162d03d2aff43ba16', 'data/raw/identity_recheck_20261108/edu_0cd62bab8d30.txt': '627c34b5633642b962d07d9c4bdabf3e437a40afec3ae75ed338e6b67dc9426a', 'data/raw/identity_recheck_20261108/edu_4b6358aa813c.txt': '2c2ee452ded643a343cab01b087449bb419d32553c287b8812b6fd461043d75a', 'data/raw/identity_recheck_20261108/edu_8828b2c22096.html': 'd0f0fb69cade5d9fab5ee5baeee3ff120dab60837bc5d80bf5833e9386b599dd', 'data/raw/identity_recheck_20261108/edu_8828b2c22096.txt': '36c448d4dd113a44192eef0029e0c0bda25c570121991120575ddea7fc33de16', 'data/raw/identity_recheck_20261108/edu_50635a884a83.html': '2210c13ce696038c367cf14537d28d2f9eb6632d54f5c44fd0af0de99cbb7e6b', 'data/raw/identity_recheck_20261108/edu_50635a884a83.txt': '12a7a6a2e25a600970d7898899a19869b095e7e66f4854940409d1b6c0c55676', 'data/raw/identity_recheck_20261108/edu_3dbf11175bbc.html': 'd8c7832cdebea6aad2286276443d655c765b7ad28bfb84788c3eefdc80e4f636', 'data/raw/identity_recheck_20261108/edu_3dbf11175bbc.txt': 'e2b6de733c4d8551000c8bd44b90e516afe76aea810e7745676f887cf32f65fc', 'data/raw/identity_recheck_20261108/edu_a4cc4bf704e9.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_6088f01cbb63.html': '2b24023cc8daeb7f6fe2ccb006cbdd56363341e7603bfbc4f14018f2db01f236', 'data/raw/identity_recheck_20261108/edu_6088f01cbb63.txt': 'd5aed18e71c704fcf905b7c6dd574018b7dd119f3db50294884a60598928595f', 'data/raw/identity_recheck_20261108/edu_a4cc4bf704e9.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_2aa85add8b63.html': '449df4af2d2be09d0a3d1b1beaaec7a0260cfec439a30e831b53cbb6f1ce46c9', 'data/raw/identity_recheck_20261108/edu_2aa85add8b63.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_821d9d4d33e0.html': 'c58ece4569bec5616b5aa95c3172c03a7bc570ac8f7333ca14da4731a7a5a13d', 'data/raw/identity_recheck_20261108/edu_5c0045a61857.html': '71bf4ac515f31d5b6cd1acf530d09e83884b6b6c9ace5f6d7e4540ef66226db7', 'data/raw/identity_recheck_20261108/edu_5c0045a61857.txt': '32d1176e279f6dc2cde1e47ecd0aee0be23aef2a9aa279890d0cf3557f207af9', 'data/raw/identity_recheck_20261108/edu_821d9d4d33e0.txt': '189f4e0926d20057fa8793cc967e46a1dc9899fd8b21b5dd64e6bbf09e920d5d', 'data/raw/identity_recheck_20261108/edu_681b86a381c9.html': '609d334c53e539485215e2c02b9de027a540d2e3a326e889f92c46cecaa66b74', 'data/raw/identity_recheck_20261108/edu_681b86a381c9.txt': '014760a31e0e33eaa0e1a8f6f564809d2d3815b5cfc6e4e7b807b7c1494c23a1', 'data/raw/identity_recheck_20261108/edu_5ec760d5cb86.txt': '63f5917ca3d49e98f9fcb3dbdff8b24495590e1d05ad0607cb5162bed009c626', 'data/raw/identity_recheck_20261108/education_current_manifest.json': '4de3c77a1959e496f370bf231ae2fb5e8a9d5a3dc31c15a525123ff5c148b417', 'data/raw/identity_recheck_20261108/current_source_reviewed_facts.json': 'afe5083ed40d2995b0d7c6907df5e2805321a6bfeacda05bb121f476a77f9ea9'})
for f,expected in PINS.items():assert sha(ROOT/f)==expected,f

PREVIOUS_OUTPUT_SHA256.update({'data/processed/clustering_all_20261108.csv': '1a6a3bda7aaa1acd047124dfdc16f889ffc02021d8c474e07be9c78eb0c54e7c', 'data/processed/clustering_mayor_20261108.csv': '48bfbc1f7e17d3ffdbfce992bf8eb14080cd55874165f0e0ffdbf0e04f62fa27', 'data/processed/clustering_council_20261108.csv': '7b6f8c16d717a534619e274847adecf8bccd44945dc1d3d12f390f2eddb7a693', 'data/processed/clustering_merge_audit_20261108.csv': '22d576db9615e276f8b73be83c68c3401929611b442767a45abd01871ca3c7ce', 'data/processed/clustering_readiness_20261108.csv': 'b758284eb71df94568c542e7aff93501f9bdd9d56a0ecfff846fd1c0e94b12ea', 'data/processed/clustering_remaining_worklist_20261108.csv': '67e8afb3d405a4fc9253c981e7bd19167355d732cc8beba68f0c9788ca43d774', 'data/processed/clustering_g6_new_evidence_20261108.csv': '27156ef1eb4c76dd53e2a57523314fe5882eb52fa5ea1a41903b340f61812579'})

# 魏造文專項查核：固定本輪來源，保留早期輸入版本。
PINS.update({'data/raw/wei_zaowen_20261108/olc_api.json': '96aa4df09ab6954f417991170d616bff8fd2f9dc7cdbb2fc2f844664c27cc2a8', 'data/raw/wei_zaowen_20261108/udn_interview.html': 'ec24d9055b5b927124a4cc1b11d970ad2f6776e3de2d0f214edaa5c4516304a2', 'data/raw/wei_zaowen_20261108/udn_interview.txt': '1b0eb40c820b8b75066bda56fd8336421a3f12eef38eaec787dce9687d809b60', 'data/raw/wei_zaowen_20261108/ebc_interview.html': 'c5b717e218146954e385eca907caf3ca61451448be4c4ab59eed8834309ddccf', 'data/raw/wei_zaowen_20261108/ebc_interview.txt': '2e83be9eb8f9d270b75c69be7a3e8d0eccecbcb540bf2ce4a6d50fab9d1805c4', 'data/raw/wei_zaowen_20261108/ltn_candidates.html': '43a4cf9fa642af205c38f80b8ed524df2eeed6ddc8727a2823eef0fbdd69cf1a', 'data/raw/wei_zaowen_20261108/ltn_candidates.txt': '6dc40a2e861329148d2f7d1af9750a076d2403202b5ee8bd8146f5a723fdf6cf', 'data/raw/wei_zaowen_20261108/blindvote_bio.html': 'cfe03d80adcdead67c8834317693e1635842d7d37d6deb21feb40eb007fd0522', 'data/raw/wei_zaowen_20261108/blindvote_bio.txt': '6b5d1f9e080957df95ca6ce673a073e6fef9a4954b3980c862ab2b2cb2ef2a9b', 'data/raw/wei_zaowen_20261108/cdn_report.html': 'a749eaf9957726aec62d7fcb68d993da1077da7271f484763832ab0c793a1208', 'data/raw/wei_zaowen_20261108/cdn_report.txt': '91695eb4fbae7b3647b2180f9304cfa0d28f1516b09c9faba207d281d5e44a68', 'data/raw/wei_zaowen_20261108/olc_profile.html': 'c939caad45422d1d2ea327a84bda4e71eab38a2d0d1e28bad91f560985857c99', 'data/raw/wei_zaowen_20261108/olc_profile.txt': 'fe32599962a2ffc80e03bd83576a59db9e90c3ecd0c8715839b546feb47b5d75', 'data/raw/wei_zaowen_20261108/manifest.json': 'b6940accafa1fb59d16525efefe586cb3dc671c6a36f6e55f73adfcbaff0aaee', 'data/raw/wei_zaowen_20261108/review.json': '5af4fbf0d3787d7bb9edd22d2d0a70620a67d30b77906532774c55873821dcfc'})
for f,expected in PINS.items():assert sha(ROOT/f)==expected,f
PREVIOUS_OUTPUT_SHA256.update({'data/processed/clustering_all_20261108.csv': '88d6247efe114956ee0f207c6f3111a16f2ef1942b6f1862bee1c0b8dfb0f64a', 'data/processed/clustering_mayor_20261108.csv': '48bfbc1f7e17d3ffdbfce992bf8eb14080cd55874165f0e0ffdbf0e04f62fa27', 'data/processed/clustering_council_20261108.csv': '233ab56df9413d425fcb307b553551c909682a97254a73490cf8204d4ce5c4aa', 'data/processed/clustering_merge_audit_20261108.csv': 'deb40bb175c637a76c726cd5defe6145da39648077019465deae2fa0a0c72317', 'data/processed/clustering_readiness_20261108.csv': '245ec1f6e56ec7eb3da0a6842fee511fbfb7d8adda3b7d616761f13f67a89499', 'data/processed/clustering_remaining_worklist_20261108.csv': 'f22dd177cb0b2e72da445d58e504a2f1161ccd7cd88dd5c38f318c03f1a49622', 'data/processed/clustering_g6_new_evidence_20261108.csv': '42fa9ef0a427affbadb455ad56caa283de82500078e34c5fa1a17d4bed4a4248'})


In [2]:
# 確認的規則只套用到已核對補查表；未研究的空白不冒稱未公開。
for source in ['data/processed/mayor_g1_g6_gap_review_20261104.csv','data/processed/council_tk_g1_g5_gap_review_20261104.csv']:
    supplement=pd.read_csv(ROOT/source,dtype=str,keep_default_na=False)
    assert not supplement.duplicated(['candidate_key','field']).any()
    for r in supplement.to_dict('records'):
        key,field=r['candidate_key'],r['field']; assert key in merged.index
        assert merged.at[key,'candidate_name']==r['candidate_name']
        col=COLS[field]; value=r['proposed_value']; old=merged.at[key,col]
        status='applied'
        if not value or not r['decision'].startswith('accepted'):status='not_proposed'
        elif field=='G2' and '照片' in r['note']:status='withheld_photo_inference'
        elif old and old!=value:status='conflict_retained_old'
        record=dict(candidate_key=key,candidate_name=r['candidate_name'],field=field,old_value=old,proposed_value=value,status=status,decision_basis=r['decision_basis'],source_table=source,source_url=r['source_url'],source_file=r['source_file'],evidence_excerpt=r['evidence_excerpt'],note=r['note'])
        updates.append(record)
        if status!='applied':continue
        merged.at[key,col]=value
        src='; '.join(x for x in [r['source_url'],r['source_file'],r.get('cec_record',''),source] if x)
        merged.at[key,field+'_source']=src
        merged.at[key,'supplement_20261108']+=';'+field
        merged.at[key,'checked_date']='2026-11-08'
        if field=='G1':
            merged.at[key,'G1_source_tier']='supplement_'+r['decision_basis']
        if field=='G3':
            merged.at[key,'G3_note']=r['note']+'；依2026-11-08張睿庠確認的公開呈現分類，非學位認證'
            merged.at[key,'G3_public_presentation_rule']=r['decision_basis']
            merged.at[key,'education_source_tier']='supplement_'+r['decision_basis']
            if re.search('未公開|未填學歷|查無任何學歷|查無.*學歷|user_rule_no_education_lowest',r['note']+' '+r['decision_basis']):
                merged.at[key,'G3_unpublished_lowest_flag']='1'
        if field=='G5':
            merged.at[key,'G5_label']='首次參選任何民選公職（含修訂規則推定）' if value=='1' else '非首次參選任何民選公職'
            merged.at[key,'G5_evidence']='; '.join(x for x in [r['evidence_excerpt'],r['note']] if x)
            merged.at[key,'G5_audit_status']='yang_revised_rule_'+r['decision_basis']
# 兩人身分再核對：不將同名網站抄錄當獨立佐證。
for name,file,rownum in [('林聖峰','112_elcand.csv',7405),('陳奕蓁','105_elcand.csv',10)]:
    key=merged[merged.candidate_name.eq(name)].index.item()
    raw=list(csv.reader((ROOT/'data/raw/cec_all_offices'/file).open(encoding='utf-8-sig')))[rownum-1]
    assert raw[6]==name
    roc=raw[9];dob=f'{int(roc[:3])+1911:04d}-{roc[3:5]}-{roc[5:7]}'
    assert int(merged.at[key,'G1_age_2026_minus_birth_year'])==2026-int(dob[:4])
    merged.at[key,'birth_date']=dob;merged.at[key,'birth_year']=dob[:4]
    merged.at[key,'identity_recheck_20261108']='career_location_party_corroborated_school_current_unavailable' if name=='林聖峰' else 'current_office_and_historical_school_corroborated'
    if name=='陳奕蓁':
        merged.at[key,'education_raw']='高美醫護管理專科學校；大仁科技大學休閒事業管理碩士'
        merged.at[key,'G3_source']+='; data/raw/identity_recheck_20261108/chen_moi.html'
    # 新增證據保留層級；既有性別不由照片推測。
# 依保存的education_raw逐人重編碼；不新增網站事實或聲稱查驗學位。
PUBLIC_EDUCATION = {'游明諺': '大學', '陳宛毓': '大學', '陳韋任': '研究所', '郭書成': '大學', '林啟明': '大學', '周曉芸': '大學', '蔡佩珊': '研究所', '簡嘉佑': '大學', '李雅芬': '大學', '謝昌年': '大學', '吳翌立': '大學以下', '詹錦章': '大學以下', '林世賢': '大學以下', '許志宏': '研究所', '蔣煙燈': '大學以下', '吳淳祐': '大學', '廖昶淳': '研究所', '林宜豊': '研究所', '何秀英': '大學以下', '陳靖雅': '大學', '劉仲書': '大學以下', '陳思伃': '大學', '董昌華': '大學以下', '林德建': '大學以下', '吳程澤': '大學', '鄭宏輝': '研究所', '陳珍裕': '大學以下', '施淑婷': '大學', '凌子楚': '研究所', '陳玥妗': '研究所', '顏翎熹': '大學', '林詩涵': '大學'}
for name,value in PUBLIC_EDUCATION.items():
    matches=merged[merged.candidate_name.eq(name)&merged.G3_education_level.eq('')]
    assert len(matches)==1,name
    key=matches.index.item();raw=merged.at[key,'education_raw'];assert raw
    updates.append(dict(candidate_key=key,candidate_name=name,field='G3',old_value='',proposed_value=value,status='recoded_stored_education',decision_basis='publicly_listed_education_rule_20261108',source_table='data/processed/candidates_g1_g6_current.csv',source_url=merged.at[key,'G3_source'],source_file='',evidence_excerpt=raw,note='逐人閱讀保存的學歷文字；列名未註明畢業可採用，明載在學／肄業不視為完成；保留來源身分限制'))
    merged.at[key,'G3_education_level']=value
    merged.at[key,'G3_public_presentation_rule']='publicly_listed_education_rule_20261108'
    merged.at[key,'G3_note']='依保存學歷文字與張睿庠公開呈現規則重編碼；非學位認證，原始來源及身分核對限制保留'
    merged.at[key,'supplement_20261108']+=';G3_stored_text'
    merged.at[key,'checked_date']='2026-11-08'

FAMILY_FACTS=json.loads((ROOT/"data/raw/identity_recheck_20261108/g6_reviewed_facts.json").read_text())
family_rows=[]
for r in FAMILY_FACTS:
    match=merged[merged.candidate_name.eq(r['candidate_name'])];assert len(match)==1
    key=match.index.item();assert merged.at[key,'G6_political_family']==''
    merged.at[key,'G6_political_family']='1';merged.at[key,'G6_label']='三親等／配偶政治家族有來源正值'
    detail=r['relative_name']+'；'+r['relationship']+'；'+r['relative_public_office']
    merged.at[key,'G6_evidence']=detail
    merged.at[key,'G6_source']=r['url']+'; data/raw/identity_recheck_20261108/g6_reviewed_facts.json'
    merged.at[key,'G6_evidence_tier']='media_full_text_relationship_and_public_office'
    merged.at[key,'checked_date']='2026-11-08';merged.at[key,'supplement_20261108']+=';G6_new_source'
    family_rows.append(dict(candidate_key=key,**r))
    updates.append(dict(candidate_key=key,candidate_name=r['candidate_name'],field='G6',old_value='',proposed_value='1',status='new_full_text_evidence',decision_basis='relationship_and_public_office',source_table='data/raw/identity_recheck_20261108/g6_reviewed_facts.json',source_url=r['url'],source_file='data/raw/identity_recheck_20261108/'+r['file'],evidence_excerpt=detail,note=r['access']+'；'+r['basis']))

# 使用者明確採「搜尋後未發現」編0；未搜尋或有待查線索者仍空白。
search_decisions=json.loads((ROOT/'data/raw/identity_recheck_20261108/g6_search_decisions.json').read_text())
merged['G6_search_status_20261108']=merged.G6_political_family.map({'1':'positive_evidence_retained','':'not_screened_this_round'})
for r in search_decisions:
    match=merged[merged.candidate_name.eq(r['candidate_name'])];assert len(match)==1
    key=match.index.item();merged.at[key,'G6_search_status_20261108']=r['status']
    if r['decision']!='0':continue
    assert merged.at[key,'G6_political_family']==''
    merged.at[key,'G6_political_family']='0';merged.at[key,'G6_label']='依指定線上搜尋未發現符合範圍的政治親屬'
    merged.at[key,'G6_evidence']=r['limitations'];merged.at[key,'G6_source']='data/raw/identity_recheck_20261108/g6_search_decisions.json'
    merged.at[key,'G6_evidence_tier']='user_defined_search_negative_not_verified_absence'
    merged.at[key,'checked_date']='2026-11-08';merged.at[key,'supplement_20261108']+=';G6_search_negative'
    updates.append(dict(candidate_key=key,candidate_name=r['candidate_name'],field='G6',old_value='',proposed_value='0',status='searched_no_qualifying_evidence',decision_basis='user_online_no_evidence_rule_20261108',source_table='data/raw/identity_recheck_20261108/g6_search_decisions.json',source_url=';'.join(r['reviewed_urls']),source_file='data/raw/identity_recheck_20261108/g6_negative_search.txt',evidence_excerpt='',note=r['limitations']))

# 當屆報導全文補查；保存新事實與原頁，不能以索引片段直接填入。
current_facts=json.loads((ROOT/'data/raw/identity_recheck_20261108/current_source_reviewed_facts.json').read_text())
for r in current_facts:
    key=r['candidate_key'];assert key in merged.index and merged.at[key,'candidate_name']==r['candidate_name']
    for field,value in [('G1',r['age_value']),('G3',r['education_value']),('G5',r['G5_value']),('G6','1' if r['family'] else '')]:
        if not value:continue
        col=COLS[field];old=merged.at[key,col];assert old in ['',value],(key,field,old,value)
        if old:continue
        merged.at[key,col]=value;merged.at[key,field+'_source']=r['url']+'; '+r['source_file']
        merged.at[key,'supplement_20261108']+=';'+field+'_current_source';merged.at[key,'checked_date']='2026-11-08'
        updates.append(dict(candidate_key=key,candidate_name=r['candidate_name'],field=field,old_value=old,proposed_value=value,status='new_current_full_text',decision_basis='current_full_text_explicit_'+field,source_table='data/raw/identity_recheck_20261108/current_source_reviewed_facts.json',source_url=r['url'],source_file=r['source_file'],evidence_excerpt='',note=r['note']))
        if field=='G1':merged.at[key,'G1_source_tier']='media_2026_reported_age_plusminus1'
        if field=='G3':
            merged.at[key,'education_raw']=' / '.join(next(x['record']['items'] for x in json.loads((ROOT/'data/raw/identity_recheck_20261108/education_current_leads.json').read_text()) if x['candidate_key']==key))
            merged.at[key,'education_source_tier']='media_current_full_text_2026'
            merged.at[key,'G3_note']=r['note'];merged.at[key,'G3_public_presentation_rule']='current_source_full_text'
        if field=='G5':
            merged.at[key,'G5_evidence']='2026報導明稱政治素人，採使用者確認修訂規則'
            merged.at[key,'G5_label']='首次參選任何民選公職（素人報導推定）';merged.at[key,'G5_audit_status']='yang_revised_rule_reported_novice'
        if field=='G6':
            relative,relationship,degree,office=r['family'];detail=relative+'；'+relationship+'；'+office
            merged.at[key,'G6_label']='三親等／配偶政治家族有來源正值';merged.at[key,'G6_evidence']=detail;merged.at[key,'G6_evidence_tier']='media_current_full_text_relationship_and_public_office'
            merged.at[key,'G6_search_status_20261108']='positive_current_full_text'
            family_rows.append(dict(candidate_key=key,candidate_name=r['candidate_name'],relative_name=relative,relationship=relationship,degree=degree,relative_public_office=office,url=r['url'],file=r['source_file'],access='HTTP200全文已讀並保存',basis=r['note']))

# 魏造文：使用者編碼決定另有文字欄位支持，不從照片外觀推斷。
wei=json.loads((ROOT/'data/raw/wei_zaowen_20261108/review.json').read_text());key=wei['candidate_key']
api=json.loads((ROOT/'data/raw/wei_zaowen_20261108/olc_api.json').read_text())['candidate']['Candidate']
assert api['name']==merged.at[key,'candidate_name']=='魏造文' and api['gender']=='m' and api['no']=='2'
assert api['birth'] is None
for path,expected in wei['historical_exact_name_scan']['files'].items():
    assert sha(ROOT/path)==expected,path
    assert '魏造文' not in (ROOT/path).read_text(encoding='utf-8-sig')
assert merged.at[key,'G1_age_2026_minus_birth_year']==''
assert merged.at[key,'G2_sex']=='' and merged.at[key,'G6_political_family']==''
merged.at[key,'G2_sex']='男'
merged.at[key,'G2_source_tier']=wei['G2']['tier']
merged.at[key,'G2_source']=wei['G2']['source']+'; data/raw/wei_zaowen_20261108/olc_api.json; 使用者本次明確編碼決定'
merged.at[key,'G6_political_family']='0'
merged.at[key,'G6_label']='依指定線上搜尋未發現符合範圍的政治親屬'
merged.at[key,'G6_evidence']=wei['G6']['note']
merged.at[key,'G6_source']='data/raw/wei_zaowen_20261108/review.json'
merged.at[key,'G6_evidence_tier']=wei['G6']['tier']
merged.at[key,'G6_search_status_20261108']='none_found_online_screen'
merged.at[key,'supplement_20261108']+=';G2_explicit_OLC;G6_search_negative'
merged.at[key,'checked_date']='2026-11-08'
for field,value,status,basis,url,sourcefile,note in [
    ('G2','男','new_explicit_secondary_field','OLC_gender_m_and_user_confirmed',wei['G2']['source'],'data/raw/wei_zaowen_20261108/olc_api.json',wei['G2']['note']),
    ('G6','0','searched_no_qualifying_evidence','user_online_no_evidence_rule_20261108','','data/raw/wei_zaowen_20261108/review.json',wei['G6']['note'])]:
    updates.append(dict(candidate_key=key,candidate_name='魏造文',field=field,old_value='',proposed_value=value,status=status,decision_basis=basis,source_table='data/raw/wei_zaowen_20261108/review.json',source_url=url,source_file=sourcefile,evidence_excerpt='gender=m' if field=='G2' else '',note=note))

merged=merged.reset_index(drop=True)
audit=pd.DataFrame(updates)
assert not audit.status.eq('conflict_retained_old').any(),audit[audit.status.eq('conflict_retained_old')]
assert merged.candidate_key.is_unique and len(merged)==len(base)
assert merged.loc[merged.G6_political_family.eq(''),'G6_political_family'].eq('').all()
assert merged.G6_political_family.eq('1').sum()==base.G6_political_family.eq('1').sum()+9
assert merged.G6_political_family.eq('0').sum()==3
assert (merged.loc[~merged.candidate_key.eq('2026-C2-072'),'G2_sex']==base.loc[~base.candidate_key.eq('2026-C2-072'),'G2_sex']).all()
assert merged.loc[merged.candidate_key.eq('2026-C2-072'),'G2_sex'].item()=='男'
for col in ['G5_first_time_any_elected_office','G6_political_family']:assert set(merged[col])<={'','0','1'}
assert set(merged.G3_education_level)<={'','大學以下','大學','研究所'}
pools={'all':merged,'mayor':merged[~merged.office_type.str.contains('議員')].copy(),'council':merged[merged.office_type.str.contains('議員')].copy()}
assert len(pools['mayor'])==81 and len(pools['council'])==1502
assert set(pools['mayor'].candidate_key).isdisjoint(set(pools['council'].candidate_key))
summary=[];gaps=[]
for pool,d in pools.items():
    summary.append(dict(pool=pool,rows=len(d),**{f+'_missing':int(d[c].eq('').sum()) for f,c in COLS.items()},complete_G1_G5=int(d[list(COLS.values())[:5]].ne('').all(axis=1).sum()),complete_G1_G6=int(d[list(COLS.values())].ne('').all(axis=1).sum())))
for r in merged.to_dict('records'):
    for f,c in COLS.items():
        if not r[c]:gaps.append(dict(candidate_key=r['candidate_key'],candidate_name=r['candidate_name'],county_city=r['county_city'],office_type=r['office_type'],field=f,status='not_yet_established',source_lead=r.get(f+'_source','')))
outputs={**{f'data/processed/clustering_{k}_20261108.csv':d for k,d in pools.items()},'data/processed/clustering_merge_audit_20261108.csv':audit,'data/processed/clustering_readiness_20261108.csv':pd.DataFrame(summary),'data/processed/clustering_remaining_worklist_20261108.csv':pd.DataFrame(gaps)}
outputs['data/processed/clustering_g6_new_evidence_20261108.csv']=pd.DataFrame(family_rows)
for f,d in outputs.items():
    dest=ROOT/f;assert dest.is_relative_to(ROOT/'data/processed')
    if dest.exists():
        prior=pd.read_csv(dest,dtype=str,keep_default_na=False)
        if not prior.equals(d.astype(str).reset_index(drop=True)):
            assert sha(dest)==PREVIOUS_OUTPUT_SHA256[f],f
            d.to_csv(dest,index=False,encoding='utf-8-sig')
    else:d.to_csv(dest,index=False,encoding='utf-8-sig')
    reread=pd.read_csv(dest,dtype=str,keep_default_na=False)
    pd.testing.assert_frame_equal(d.astype(str).reset_index(drop=True),reread,check_dtype=False)
for f,expected in PINS.items():assert sha(ROOT/f)==expected
verification={f:dict(rows=len(d),columns=list(d.columns),sha256=sha(ROOT/f)) for f,d in outputs.items()}
print(json.dumps(verification,ensure_ascii=False,indent=2))
display(audit.groupby(['field','status']).size().rename('cells').to_frame())
display(pd.DataFrame(summary))


{
  "data/processed/clustering_all_20261108.csv": {
    "rows": 1583,
    "columns": [
      "candidate_key",
      "candidate_name",
      "county_city",
      "office_type",
      "electoral_district",
      "G1_age_2026_minus_birth_year",
      "birth_date",
      "birth_year",
      "G1_source_tier",
      "G1_source",
      "G2_sex",
      "G2_source_tier",
      "G2_source",
      "G3_education_level",
      "education_raw",
      "G3_source",
      "G3_note",
      "education_source_tier",
      "education_historical_level",
      "G4_party",
      "party_registration_raw",
      "G4_source",
      "G4_note",
      "G5_first_time_any_elected_office",
      "G5_label",
      "G5_prior_election_year_example",
      "G5_evidence",
      "G5_source",
      "G5_audit_status",
      "G6_political_family",
      "G6_label",
      "G6_evidence",
      "G6_source",
      "G6_evidence_tier",
      "source_conflict_flag",
      "checked_date",
      "G3_public_presentation_rule",
      "G3

cells
field status                                
G1    applied                             33
      new_current_full_text                5
      not_proposed                        38
G2    new_explicit_secondary_field         1
      withheld_photo_inference             1
G3    applied                             51
      new_current_full_text               16
      not_proposed                        30
      recoded_stored_education            32
G5    applied                             86
      new_current_full_text                1
      not_proposed                         4
G6    new_current_full_text                5
      new_full_text_evidence               4
      searched_no_qualifying_evidence      3

,pool,rows,G1_missing,G2_missing,G3_missing,G4_missing,G5_missing,G6_missing,complete_G1_G5,complete_G1_G6
0,all,1583,688,0,626,0,640,1478,872,87
1,mayor,81,1,0,0,0,0,57,80,23
2,council,1502,687,0,626,0,640,1421,792,64
